### Results

## Download Historical Data from Broker API

Fetch historical OHLC+ data for extracted stocks using Shoonya broker API


In [ ]:
import logging
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import yaml
import pyotp
from NorenRestApiPy.NorenApi import NorenApi

print("=" * 80)
print("STEP 9: Download Historical Data from Broker API")
print("=" * 80)
print()

# ==================== EXTRACTED STOCKS ====================
print("📊 EXTRACTED STOCKS FOR DOWNLOAD")
print("=" * 80)
print(f"Total stocks to download: {len(symbols_array)}")
print()
print("Stock List:")
for i, stock in enumerate(symbols_array, 1):
    print(f"  {i}. {stock}")
print()

# ==================== API CONFIGURATION ====================
print("=" * 80)
print("🔐 INITIALIZING BROKER API")
print("=" * 80)

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        super().__init__(host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')

try:
    # Initialize API
    api = ShoonyaApiPy()
    print("✓ API class initialized")
    
    # Load credentials
    with open('cred.yml') as f:
        cred = yaml.load(f, Loader=yaml.FullLoader)
    print("✓ Credentials loaded from cred.yml")
    
    # Login
    TOKEN = cred['factor2']
    otp = pyotp.TOTP(TOKEN).now()
    ret = api.login(
        userid=cred['user'],
        password=cred['pwd'],
        twoFA=otp,
        vendor_code=cred['vc'],
        api_secret=cred['apikey'],
        imei=cred['imei']
    )
    
    if ret:
        print("✓ Login Successful")
    else:
        print("✗ Login Failed - Exiting")
        raise Exception("API Login Failed")
    
    print()
    
    # ==================== PREPARE DATA DOWNLOAD ====================
    print("=" * 80)
    print("📅 PREPARING DATA DOWNLOAD")
    print("=" * 80)
    
    # Calculate date range (4 months back)
    now = datetime.now()
    start_date = now - relativedelta(months=4)
    start_date = start_date.replace(hour=0, minute=0, second=0, microsecond=0)
    start_timestamp = start_date.timestamp()
    
    print(f"Start date: {datetime.fromtimestamp(start_timestamp)}")
    print(f"End date: {now}")
    print(f"Data period: 4 months")
    print(f"Exchange: NSE")
    print(f"Interval: 1 minute")
    print()
    
    # ==================== DATA OUTPUT DIRECTORY ====================
    output_directory = r'D:\AlgoRepo\ShoonyaAPI_Code\Testing_Use\Stocks_DATA'
    print(f"Output directory: {output_directory}")
    print()
    
    # ==================== DOWNLOAD PROGRESS ====================
    print("=" * 80)
    print("📥 DOWNLOADING HISTORICAL DATA")
    print("=" * 80)
    print()
    
    successful_downloads = []
    failed_downloads = []
    
    for idx, stock in enumerate(symbols_array, 1):
        try:
            print(f"[{idx}/{len(symbols_array)}] Downloading {stock}...", end=" ")
            
            # Fetch the data from the API
            ret = api.get_time_price_series(exchange='NSE', token=stock, starttime=start_timestamp, interval=1)
            
            if ret and len(ret) > 0:
                # Convert the response into a DataFrame
                df_stock = pd.DataFrame(ret)
                
                # Save the DataFrame to an Excel file
                output_file_path = f'{output_directory}\\{stock}.xlsx'
                df_stock.to_excel(output_file_path, index=False)
                
                print(f"✓ ({len(df_stock)} rows saved)")
                successful_downloads.append(stock)
                
            else:
                print(f"✗ (No data returned)")
                failed_downloads.append(stock)
                
        except Exception as e:
            print(f"✗ (Error: {str(e)[:50]})")
            failed_downloads.append(stock)
            logging.error(f"Error downloading {stock}: {e}")
    
    print()
    
    # ==================== DOWNLOAD SUMMARY ====================
    print("=" * 80)
    print("✅ DOWNLOAD COMPLETE - SUMMARY REPORT")
    print("=" * 80)
    print()
    
    print(f"📊 STATISTICS:")
    print(f"   Total stocks: {len(symbols_array)}")
    print(f"   Successfully downloaded: {len(successful_downloads)}")
    print(f"   Failed downloads: {len(failed_downloads)}")
    print(f"   Success rate: {(len(successful_downloads)/len(symbols_array)*100):.1f}%")
    print()
    
    if successful_downloads:
        print(f"✓ SUCCESSFUL DOWNLOADS ({len(successful_downloads)}):")
        for i, stock in enumerate(successful_downloads, 1):
            print(f"   {i}. {stock}")
        print()
    
    if failed_downloads:
        print(f"✗ FAILED DOWNLOADS ({len(failed_downloads)}):")
        for i, stock in enumerate(failed_downloads, 1):
            print(f"   {i}. {stock}")
        print()
    
    print("=" * 80)
    print(f"📁 Files saved to: {output_directory}")
    print(f"🎯 All data ready for backtesting!")
    print("=" * 80)
    
except FileNotFoundError:
    print("✗ Error: cred.yml file not found!")
    print("   Please ensure cred.yml exists in the current directory")
except Exception as e:
    print(f"✗ Error: {str(e)}")
    print("   Please check your API credentials and connection")


In [ ]:
import pandas as pd
import os
from io import StringIO

print("=" * 80)
print("AUTOMATIC DATA EXTRACTION - EXCEL TO stocks_info FORMAT")
print("=" * 80)
print()

# ==================== EXTRACTION FUNCTION ====================
def extract_from_text(text_data):
    """Convert text data (tab or comma separated) to stocks_info format"""
    try:
        # Try reading as tab-separated (Excel copy format)
        df = pd.read_csv(StringIO(text_data), sep='\t', header=None)
    except:
        try:
            # Try comma-separated
            df = pd.read_csv(StringIO(text_data), sep=',', header=None)
        except:
            print("✗ Could not parse data. Please paste tab or comma-separated data.")
            return None
    
    # Extract to stocks_info format
    stocks_info = []
    for idx, row in df.iterrows():
        try:
            date_time = str(row.iloc[0]).strip()  # First column
            stock_symbol = str(row.iloc[1]).strip()  # Second column
            stocks_info.append((date_time, stock_symbol))
        except:
            continue
    
    return stocks_info


def extract_from_csv_file(file_path):
    """Load stocks_info directly from CSV file"""
    try:
        df = pd.read_csv(file_path)
        stocks_info = []
        for idx, row in df.iterrows():
            date_time = str(row.iloc[0]).strip()  # First column (date)
            stock_symbol = str(row.iloc[1]).strip()  # Second column (symbol)
            stocks_info.append((date_time, stock_symbol))
        return stocks_info
    except Exception as e:
        print(f"✗ Error reading CSV: {e}")
        return None


# ==================== AUTO-EXTRACT FROM CLIPBOARD ====================
try:
    import pyperclip
    print("📋 Attempting to extract from clipboard...")
    print()
    
    clipboard_data = pyperclip.paste()
    stocks_info = extract_from_text(clipboard_data)
    
    if stocks_info and len(stocks_info) > 0:
        print(f"✓ Successfully extracted {len(stocks_info)} stocks from clipboard!")
        print()
        print("stocks_info = [")
        for date_time, symbol in stocks_info:
            print(f"    ('{date_time}', '{symbol}'),")
        print("]")
        print()
    else:
        print("⚠️  Clipboard is empty or no data extracted.")
        print()
        
except ImportError:
    print("⚠️  pyperclip not available. Using CSV file extraction instead.")
    print()
    
    # Try loading from CSV file
    csv_path = r'C:\Users\omkar\Downloads\filtered_stocks.csv'
    if os.path.exists(csv_path):
        stocks_info = extract_from_csv_file(csv_path)
        if stocks_info and len(stocks_info) > 0:
            print(f"✓ Successfully loaded {len(stocks_info)} stocks from {csv_path}")
            print()
            print("stocks_info = [")
            for date_time, symbol in stocks_info:
                print(f"    ('{date_time}', '{symbol}'),")
            print("]")
            print()
    else:
        print(f"⚠️  CSV file not found at {csv_path}")
        print("   Please copy your data to clipboard and run this cell again.")
        print("   Or ensure filtered_stocks.csv exists.")
        stocks_info = []

print("=" * 80)
print(f"📊 Total stocks loaded: {len(stocks_info)}")
print("=" * 80)
print()

In [ ]:

#this is code whre you need to paste in ('13-09-2024 10:00', 'MANAPPURAM'), this way and make sure all 
# the stocks are placed in folder whih will give the output with backtest .

import pandas as pd
import os

# Set the directory where the Excel sheets are stored
excel_directory = 'D:\\AlgoRepo\\ShoonyaAPI_Code\\Testing_Use\\Stocks_DATA'

# Stock info in format: (entry time, stock symbol)
stocks_info = [
    ('2026-03-20 10:00:00', 'MRPL'),
    ('2026-03-20 10:00:00', 'FINCABLES'),
    ('2026-03-20 10:00:00', 'UNIONBANK'),
    ('2026-03-24 09:45:00', 'CANFINHOME'),
    ('2026-03-24 10:00:00', 'UNOMINDA'),
    ('2026-03-24 10:00:00', 'TIINDIA'),
    ('2026-03-24 10:45:00', 'BLS'),
    ('2026-03-25 10:00:00', 'POLICYBZR'),
]

# Result storage
results = []

# Initial investment per stock
initial_investment = 200000

# Define 15:15 time for cutting positions
cutoff_time = pd.to_datetime('2026-03-25 15:15', format='%Y-%m-%d %H:%M')

# Iterate over each stock and its respective entry time
for entry_time_str, stock_name in stocks_info:

    # Files are named with -EQ suffix (e.g., MRPL-EQ.xlsx)
    excel_file_name = f"{stock_name}-EQ.xlsx"
    excel_file_path = os.path.join(excel_directory, excel_file_name)

    try:
        # Load stock data
        stock_data = pd.read_excel(excel_file_path, usecols=['time', 'intc'])
        stock_data['time'] = pd.to_datetime(stock_data['time'], format='%d-%m-%Y %H:%M:%S', errors='coerce')

        # Sort the data, ensuring latest data is processed correctly
        stock_data.sort_values(by='time', ascending=True, inplace=True)

        # Convert entry_time_str to timestamp
        entry_time = pd.to_datetime(entry_time_str, format='%Y-%m-%d %H:%M:%S')

        # Add 2 minutes to the entry time for testing
        testing_start_time = entry_time + pd.Timedelta(minutes=2)

        # Use .iloc[0] to get first row after testing_start_time
        entry_rows = stock_data[stock_data['time'] >= testing_start_time]
        
        if not entry_rows.empty:
            entry_row = entry_rows.iloc[0]
            entry_price = entry_row['intc']
            qty = int(initial_investment / entry_price)  # Calculate quantity of stocks

            # For SHORT SELLING
            profit_target = entry_price * 0.985  # Target at 0.7% LOWER (profit on short)
            stop_loss = entry_price * 1.01     # Stop loss at 1.5% HIGHER (protect from rise)

            # Filter subsequent data (after the entry row)
            subsequent_data = stock_data[stock_data['time'] > entry_row['time']]

            stop_or_tgt_hit = False
            for index, row in subsequent_data.iterrows():
                current_price = row['intc']
                current_time = row['time']

                # Check profit target first (price BELOW entry = profit on short)
                if current_price < profit_target:
                    profit_loss_amount = ((entry_price - current_price) * qty)
                    results.append([stock_name, entry_time, current_time, current_price, 'Profit', profit_loss_amount])
                    print(f"Profit target hit at {current_time}: {current_price:.2f}, Profit: {profit_loss_amount:.2f}")
                    stop_or_tgt_hit = True
                    break

                # Check stop loss (price ABOVE entry = loss on short)
                elif current_price > stop_loss:
                    profit_loss_amount = ((entry_price - current_price) * qty)
                    results.append([stock_name, entry_time, current_time, current_price, 'Loss', profit_loss_amount])
                    print(f"Stop loss hit at {current_time}: {current_price:.2f}, Loss: {profit_loss_amount:.2f}")
                    stop_or_tgt_hit = True
                    break

            # Ensure the trade is closed at cutoff time if no stop-loss or target is hit
            cutoff_rows = stock_data[stock_data['time'] >= cutoff_time]
            if not stop_or_tgt_hit and not cutoff_rows.empty:
                cutoff_price = cutoff_rows.iloc[0]['intc']
                profit_loss_amount = ((entry_price - cutoff_price) * qty)
                status = 'Profit' if cutoff_price < entry_price else 'Loss'
                results.append([stock_name, entry_time, cutoff_time, cutoff_price, f'Cut at 15:15 ({status})', profit_loss_amount])
                print(f"No SL or TGT hit. Position closed at {cutoff_time}: {cutoff_price:.2f}, PnL: {profit_loss_amount:.2f}")
            elif not stop_or_tgt_hit and cutoff_rows.empty:
                print(f"No data found for {stock_name} at the cutoff time (15:15).")

        else:
            print(f"No entry found for {stock_name} at {testing_start_time}")

    except Exception as e:
        print(f"An error occurred for {stock_name}: {e}")

# Create a DataFrame for the results
results_df = pd.DataFrame(results, columns=['Stock Name', 'Entry Time', 'Hit/Exit Time', 'Price', 'Status', 'Profit/Loss Amount'])

# Save the results to an Excel file
output_file_path = 'C:\\Users\\omkar\\Downloads\\stock_results_New_0930TO11_spec.xlsx'
results_df.to_excel(output_file_path, index=False)

print(f"Results saved to {output_file_path}")
print(f"Total results: {len(results)}")

# ==================== RESULTS SUMMARY ====================
print("\n" + "=" * 80)
print("📊 BACKTEST RESULTS SUMMARY")
print("=" * 80)
print()

total_profit_loss = results_df['Profit/Loss Amount'].sum()
winning_trades = len(results_df[results_df['Profit/Loss Amount'] > 0])
losing_trades = len(results_df[results_df['Profit/Loss Amount'] < 0])
total_trades = len(results_df)
win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0

print(f"💰 TOTAL PROFIT/LOSS: Rs. {total_profit_loss:,.2f}")
print(f"✅ WINNING TRADES: {winning_trades}")
print(f"❌ LOSING TRADES: {losing_trades}")
print(f"📈 TOTAL TRADES: {total_trades}")
print(f"🎯 WIN RATE: {win_rate:.1f}%")
print()
print("=" * 80)
print(f"📁 Files saved to: {output_file_path}")
print("=" * 80)

print(f"\nResults saved to {output_file_path}")
